# 統計模型與損失函數

對應 DLI：01、02。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/05_statistical_models_loss.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

## 學習目標與先備知識

先讀條件機率與標準化。需要理解函數是「輸入數值得到輸出」；本課補充指數與對數，不要求先學最佳化推導。



## 1. 從猜一個數字到猜一個類別

**來源摘要 [S5](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)：**線性模型結合特徵與係數；邏輯斯迴歸用於分類，可以產生類別機率。名稱中雖有「迴歸」，用途仍可為分類。

本教材自編例子：用影像亮度 x 預測某一類手勢。先算分數 $z=b+wx$，w 決定輸入變化的影響，b 是截距。若直接把 z 當機率，可能得到負數或超過 1；因此二元分類可用 sigmoid：

$$p=\frac{1}{1+e^{-z}}$$

z=0 時 p=0.5；z 很大時 p 接近 1，z 很負時 p 接近 0。e 約為 2.718，是自然指數的底數。



## 2. 真實答案、預測與誤差

| 記號 | 意思 |
|---|---|
| $x_i$ | 第 i 筆輸入特徵 |
| $y_i$ | 真實答案 |
| $\hat{y}_i$ | 預測值或預測類別，需交代定義 |
| $p_i$ | 第 i 筆屬於正類的預測機率 |
| $L$ | 損失，衡量預測和答案的不一致 |

若預測的是連續數值，可用平均平方誤差：

$$MSE=\frac1n\sum_i(y_i-\hat y_i)^2$$

**自編手算：**真實延遲 [10,20]，預測 [12,16]，誤差 [−2,4]，平方 [4,16]，平均為 10 ms²。損失是 10，不是「準確率 10%」。



## 3. 為什麼分類會用交叉熵？

**來源摘要 [S6](https://scikit-learn.org/stable/modules/model_evaluation.html)：**log loss／交叉熵使用機率預測。二元分類中，單筆損失可寫成 $-[y\ln p+(1-y)\ln(1-p)]$；多類別則關注真實類別被分配的機率。

先看答案為正類 y=1 時，公式變為 −ln p。ln 是自然對數：ln(1)=0，當 p 越接近 0，−ln p 越大。

本教材自編對照：

| 對真實類別給的機率 | 損失 $-\ln p$ | 直覺 |
|---:|---:|---|
| 0.9 | 約 0.105 | 給對答案高分 |
| 0.5 | 約 0.693 | 不確定 |
| 0.1 | 約 2.303 | 給對答案低分 |

不要把機率 0 或 1 直接帶入未保護的 log 程式，因為 ln(0) 不有限。套件通常以數值保護處理，神經網路也常直接使用 logits 的穩定損失函數。



## 4. 相同 Accuracy，損失可能差很多

本教材程式的真實類別是 [0,1]；兩組正類機率分別為 [0.6,0.6] 與 [0.99,0.99]。以 0.5 為閾值，兩者都預測 [1,1]，所以 Accuracy 都是 50%。

第一組平均交叉熵約 0.714；第二組約 2.308。第二組對第一筆錯誤更自信，受到較大懲罰。請把手算結果對照 log_loss 輸出。



## 5. 模型怎麼學？怎麼連到 DLI？

模型學習是調整參數使訓練損失下降。梯度可先理解成「損失對某參數變動的局部敏感度」；以學習率控制每次調整幅度。下降的訓練損失不保證未見資料表現提升，仍要看驗證集。

下方使用 scikit-learn 的 8×8 digits 建立輕量基準，**不是 MNIST**。連到 DLI 時，請比較輸入形狀、輸出類別數與損失函數，避免把本例的分數當作 DLI 模型結果。

## 自我檢核

1. y=0，正類機率 p=0.9 時，應代入 −ln(0.9) 還是 −ln(0.1)？
2. Accuracy 不變時，交叉熵為何仍可能改善？
3. 訓練損失降低是否足以證明模型更適合部署？

### 參考解答

1. −ln(0.1)，因真實類別是負類。
2. 分類是否跨過閾值沒有改變，但給真實類別的機率可能更合適。
3. 否。還需驗證泛化、資料代表性及部署需求。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
from sklearn.metrics import log_loss, accuracy_score
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
truth = np.array([0, 1])
for name, prob in [('moderate', [0.6, 0.6]), ('high confidence', [0.99, 0.99])]:
    prob = np.array(prob)
    print(name, 'accuracy=', accuracy_score(truth, prob >= .5),
          'cross entropy=', log_loss(truth, np.column_stack([1-prob, prob])))
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)
print('Digits baseline accuracy:', model.score(X_test, y_test))

## 如何解讀

scikit-learn digits 是 8×8 的資料，並非 DLI 使用的 MNIST；此處用它避免下載與 GPU 需求。這是基準示範，測試集不能反覆用於挑設定。

## 練習

把一個預測機率從 0.6 改成 0.99，觀察正確與錯誤時交叉熵的不同。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S5**：[scikit-learn，Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)。
- **S6**：[scikit-learn，Metrics and scoring（Log loss、Precision、Recall）](https://scikit-learn.org/stable/modules/model_evaluation.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
